# 🤖 Top 20 Gen AI Interview Questions — 2 Year Experience Level
> Covers: LLMs, RAG, Embeddings, LangChain, LangGraph, Agents, Evaluation, Prompting, Fine-tuning & more

---

## Setup — Install Dependencies

In [ ]:
# Run this cell first to install all required packages
!pip install langchain langchain-community langchain-openai openai faiss-cpu tiktoken sentence-transformers transformers torch -q

---
## Q1. What is the difference between Semantic Search and Keyword Search?

**Answer:**
- **Keyword Search** (BM25/TF-IDF): Matches exact words. Fast, but misses synonyms and intent.
- **Semantic Search**: Uses embeddings to compare meaning. Finds conceptually similar results even without exact word matches.
- **Hybrid Search**: Combines both for best accuracy.

**Interview Tip:** Mention that semantic search powers the retrieval step in RAG pipelines.

In [1]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

query = "How do I cancel my subscription?"
docs = [
    "Steps to unsubscribe from the service",       # semantic match
    "Cancel button is on the settings page",        # partial match
    "Our pricing plans start at $10/month",         # unrelated
]

query_emb = model.encode(query, convert_to_tensor=True)
doc_embs = model.encode(docs, convert_to_tensor=True)

scores = util.cos_sim(query_emb, doc_embs)[0]

for doc, score in zip(docs, scores):
    print(f"Score: {score:.4f} | {doc}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\sande\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sande\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Score: 0.6898 | Steps to unsubscribe from the service
Score: 0.5183 | Cancel button is on the settings page
Score: 0.2435 | Our pricing plans start at $10/month


---
## Q2. Explain RAG (Retrieval-Augmented Generation). How does it work?

**Answer:**
RAG = Retrieve relevant context from a vector store → Inject into LLM prompt → Generate grounded answer.

**Flow:**
```
User Query → Embed Query → Vector DB Search → Top-K Chunks → LLM Prompt → Answer
```

**Why RAG?** Reduces hallucination. Lets LLM use private/updated data without retraining.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Simulated knowledge base
texts = [
    "Our return policy allows returns within 30 days of purchase.",
    "Premium members get free shipping on all orders.",
    "To cancel your order, visit the Orders page and click Cancel.",
    "Customer support is available 24/7 via live chat.",
]
docs = [Document(page_content=t) for t in texts]

# Embed and store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

# Retrieve
query = "How do I cancel my order?"
results = vectorstore.similarity_search(query, k=2)

print(f"Query: {query}\n")
for i, r in enumerate(results):
    print(f"Result {i+1}: {r.page_content}")

---
## Q3. What are Embeddings? How are they generated?

**Answer:**
Embeddings are dense numerical vectors that encode the semantic meaning of text. Similar texts have vectors that are close in high-dimensional space (measured by cosine similarity).

Generated by encoder models like:
- `text-embedding-ada-002` (OpenAI)
- `all-MiniLM-L6-v2` (HuggingFace, free)
- `bge-large-en` (BAAI, state-of-art open source)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "The cat sat on the mat",
    "A feline rested on a rug",    # semantically similar
    "Machine learning is complex", # unrelated
]

embeddings = model.encode(sentences)
print(f"Embedding dimension: {embeddings.shape[1]}")

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\nSim(sent1, sent2) = {cosine_sim(embeddings[0], embeddings[1]):.4f}  ← High (similar meaning)")
print(f"Sim(sent1, sent3) = {cosine_sim(embeddings[0], embeddings[2]):.4f}  ← Low (different topic)")

---
## Q4. What is Chunking? What strategies do you use?

**Answer:**
Chunking = splitting documents into smaller pieces before embedding.

| Strategy | Best For |
|---|---|
| Fixed-size | Quick prototypes |
| Recursive Character | General text (most common) |
| Semantic Chunking | High accuracy RAG |
| Markdown/HTML splitter | Structured docs |

**Key params:** `chunk_size` (token count), `chunk_overlap` (context continuity)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text = """
LangChain is a framework for building LLM-powered applications.
It provides tools for chaining prompts, memory, and external data sources.
RAG pipelines use LangChain to retrieve context from vector stores.
LangGraph extends LangChain with stateful, graph-based agent workflows.
These tools together enable powerful agentic AI systems.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_text(text)
print(f"Total chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk.strip()}\n")

---
## Q5. What is Prompt Engineering? Explain key techniques.

**Answer:**
Prompt engineering = crafting inputs to get reliable, high-quality outputs from LLMs.

**Key techniques:**
- **Zero-shot**: Direct instruction, no examples
- **Few-shot**: 2-5 examples in prompt
- **Chain-of-Thought (CoT)**: "Think step by step"
- **System prompt**: Set persona/constraints
- **ReAct**: Reasoning + Acting (used in agents)

In [ ]:
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

# --- Zero-shot ---
zero_shot = PromptTemplate(
    input_variables=["text"],
    template="Classify the sentiment of this review as Positive/Negative/Neutral:\n{text}"
)
print("Zero-shot prompt:")
print(zero_shot.format(text="The product quality was okay but delivery was too slow."))

print("\n" + "="*60 + "\n")

# --- Few-shot ---
examples = [
    {"review": "Amazing product, love it!", "sentiment": "Positive"},
    {"review": "Terrible, broke in 2 days.", "sentiment": "Negative"},
]
example_template = PromptTemplate(
    input_variables=["review", "sentiment"],
    template="Review: {review}\nSentiment: {sentiment}"
)
few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="Classify the sentiment:",
    suffix="Review: {review}\nSentiment:",
    input_variables=["review"]
)
print("Few-shot prompt:")
print(few_shot.format(review="Decent quality but overpriced."))

---
## Q6. What is an LLM Agent? How does it differ from a simple LLM call?

**Answer:**
- **LLM Call**: Single prompt → single response. Stateless.
- **Agent**: LLM decides which tools to call, in what order, to complete a goal. Has a reasoning loop (ReAct pattern).

**Components:** LLM (brain) + Tools (actions) + Memory (state) + Planner (orchestration)

In [ ]:
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate

# Define tools the agent can use
@tool
def get_order_status(order_id: str) -> str:
    """Returns the status of an order given its ID."""
    mock_data = {
        "ORD001": "Shipped - Expected delivery: Tomorrow",
        "ORD002": "Processing - Will ship in 2 days",
        "ORD003": "Delivered on 2024-12-01"
    }
    return mock_data.get(order_id, "Order not found")

@tool
def calculate_refund(order_id: str) -> str:
    """Calculates refund amount for a given order."""
    return f"Refund of ₹1,499 eligible for order {order_id}"

tools = [get_order_status, calculate_refund]

# Show tool schemas
print("Available Agent Tools:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

# Demonstrate direct tool call (simulating agent action)
print("\nSimulated tool calls:")
print(get_order_status.invoke("ORD001"))
print(calculate_refund.invoke("ORD002"))

---
## Q7. What is LangGraph? When would you use it over LangChain?

**Answer:**
LangGraph builds **stateful, cyclic** multi-agent workflows as a graph (nodes + edges).

| | LangChain | LangGraph |
|---|---|---|
| Flow | Linear chains | Cyclic graphs |
| State | Limited | Full state machine |
| Use case | Simple RAG/chains | Multi-agent, loops, branching |

**Use LangGraph when:** You need loops, conditional routing, human-in-the-loop, or multiple coordinating agents.

In [ ]:
# LangGraph-style state machine (conceptual implementation)
from typing import TypedDict, Literal

# Define state schema
class AgentState(TypedDict):
    query: str
    intent: str
    answer: str
    needs_escalation: bool

# Define node functions
def classify_intent(state: AgentState) -> AgentState:
    query = state["query"].lower()
    if "refund" in query or "cancel" in query:
        state["intent"] = "support"
    elif "price" in query or "buy" in query:
        state["intent"] = "sales"
    else:
        state["intent"] = "general"
    return state

def handle_support(state: AgentState) -> AgentState:
    state["answer"] = "Your refund will be processed in 3-5 business days."
    state["needs_escalation"] = False
    return state

def handle_sales(state: AgentState) -> AgentState:
    state["answer"] = "Our premium plan starts at ₹999/month."
    state["needs_escalation"] = False
    return state

# Router (edge condition)
def route_by_intent(state: AgentState) -> Literal["support", "sales", "general"]:
    return state["intent"]

# Simulate graph execution
def run_graph(query: str):
    state: AgentState = {"query": query, "intent": "", "answer": "", "needs_escalation": False}
    state = classify_intent(state)
    route = route_by_intent(state)
    if route == "support":
        state = handle_support(state)
    elif route == "sales":
        state = handle_sales(state)
    else:
        state["answer"] = "Please contact support@company.com for help."
    return state

# Test
for q in ["I want a refund for my order", "What is your pricing?", "Hello there"]:
    result = run_graph(q)
    print(f"Query : {q}")
    print(f"Intent: {result['intent']} | Answer: {result['answer']}\n")

---
## Q8. What is the difference between Fine-tuning and RAG?

**Answer:**

| | Fine-tuning | RAG |
|---|---|---|
| Updates weights? | ✅ Yes | ❌ No |
| Needs training data | ✅ Labeled pairs | ❌ Just documents |
| Knowledge updatable? | ❌ Retrain needed | ✅ Just update vector DB |
| Cost | High | Low |
| Best for | Style/behavior/domain tone | Dynamic knowledge retrieval |

**Rule of thumb:** Use RAG first. Fine-tune only for tone/style adaptation.

In [ ]:
# Illustrate the conceptual difference

print("=" * 55)
print("FINE-TUNING Flow")
print("=" * 55)
fine_tune_flow = [
    "1. Collect (prompt, ideal_response) pairs",
    "2. Format as JSONL training file",
    "3. Submit to OpenAI / HuggingFace trainer",
    "4. New model weights saved",
    "5. Deploy fine-tuned model",
    "6. (Repeat if knowledge changes)"
]
for step in fine_tune_flow:
    print(f"  {step}")

print("\n" + "=" * 55)
print("RAG Flow")
print("=" * 55)
rag_flow = [
    "1. Chunk and embed your documents",
    "2. Store in vector DB (FAISS/Pinecone/Weaviate)",
    "3. At query time: embed query",
    "4. Retrieve top-K similar chunks",
    "5. Inject chunks into LLM prompt",
    "6. LLM generates grounded answer",
    "(Update: just re-embed new docs, no retraining!)"
]
for step in rag_flow:
    print(f"  {step}")

---
## Q9. What is Hallucination in LLMs? How do you reduce it?

**Answer:**
Hallucination = LLM confidently generating false or made-up information.

**Reduction strategies:**
1. **RAG** — ground answers in retrieved facts
2. **Temperature = 0** — deterministic output
3. **Self-consistency** — sample N outputs, pick majority
4. **Citation enforcement** — prompt LLM to only use provided context
5. **Factual verification** — use a second LLM to check

In [ ]:
# Demonstrate citation-enforcement prompting (anti-hallucination)

def build_anti_hallucination_prompt(context: str, question: str) -> str:
    return f"""You are a factual assistant. Answer ONLY using the provided context.
If the answer is not in the context, say "I don't have enough information to answer this."
Do NOT make up any information.

CONTEXT:
{context}

QUESTION: {question}

ANSWER (based only on context above):"""

context = """Our SLA guarantees 99.9% uptime. Support tickets are resolved within 24 hours.
Enterprise plans include dedicated account managers."""

questions = [
    "What is the guaranteed uptime?",
    "Do you offer a free trial?"  # not in context → should refuse
]

for q in questions:
    prompt = build_anti_hallucination_prompt(context, q)
    print(f"Q: {q}")
    print(f"Prompt snippet: ...{prompt[-200:]}")
    print()

---
## Q10. What are Vector Databases? Compare FAISS vs Pinecone vs ChromaDB.

**Answer:**
Vector DBs store and search high-dimensional embeddings efficiently using ANN (Approximate Nearest Neighbor) algorithms.

| | FAISS | ChromaDB | Pinecone |
|---|---|---|---|
| Type | Library | Local DB | Managed cloud |
| Persistence | Manual | ✅ Built-in | ✅ Built-in |
| Scale | Millions | Thousands | Billions |
| Cost | Free | Free | Paid |
| Best for | Prototyping | Dev/test | Production |

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Documents to index
documents = [
    "Python is great for data science and AI",
    "LangChain helps build LLM applications",
    "FAISS enables fast vector similarity search",
    "OpenAI GPT-4 is a powerful language model",
    "Transformers are the backbone of modern NLP",
]

# Encode and build FAISS index
embeddings = model.encode(documents).astype('float32')
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)  # Inner Product (cosine after normalize)
faiss.normalize_L2(embeddings)
index.add(embeddings)

print(f"FAISS index built with {index.ntotal} vectors (dim={dimension})\n")

# Query
query = "What library is used for vector search?"
query_emb = model.encode([query]).astype('float32')
faiss.normalize_L2(query_emb)

scores, indices = index.search(query_emb, k=3)
print(f"Query: {query}\n")
for score, idx in zip(scores[0], indices[0]):
    print(f"  Score: {score:.4f} → {documents[idx]}")

---
## Q11. Explain the ReAct Pattern in AI Agents.

**Answer:**
ReAct = **Re**asoning + **Act**ing. The LLM alternates between:
- **Thought**: Reasoning about the current state
- **Action**: Calling a tool
- **Observation**: Reading tool output

This loop continues until the LLM generates a **Final Answer**.

In [ ]:
# Simulate a ReAct reasoning trace

react_trace = [
    {
        "step": "Thought",
        "content": "The user wants to know if order ORD001 is eligible for a refund. I need to first check the order status."
    },
    {
        "step": "Action",
        "content": "get_order_status(order_id='ORD001')"
    },
    {
        "step": "Observation",
        "content": "Order ORD001 status: Delivered on 2024-11-15 (within 30-day return window)"
    },
    {
        "step": "Thought",
        "content": "Order is within the return window. Now I should calculate the refund amount."
    },
    {
        "step": "Action",
        "content": "calculate_refund(order_id='ORD001')"
    },
    {
        "step": "Observation",
        "content": "Refund of ₹1,499 eligible for order ORD001"
    },
    {
        "step": "Final Answer",
        "content": "Yes, your order ORD001 is eligible for a refund of ₹1,499 as it is within the 30-day return window."
    }
]

print("ReAct Reasoning Trace")
print("=" * 55)
for entry in react_trace:
    icon = {"Thought": "🧠", "Action": "⚡", "Observation": "👁️", "Final Answer": "✅"}.get(entry["step"], "")
    print(f"\n{icon} [{entry['step']}]")
    print(f"   {entry['content']}")

---
## Q12. What is Memory in LLM Applications? Types of Memory?

**Answer:**
LLMs are stateless — memory is how we persist context across turns.

| Memory Type | Description | Best For |
|---|---|---|
| Buffer Memory | Full chat history | Short conversations |
| Window Memory | Last N messages | Longer conversations |
| Summary Memory | Compressed summary | Very long sessions |
| Vector Memory | Semantic search of past | Episodic/long-term |

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# Window Memory: keeps last k=3 exchanges
memory = ConversationBufferWindowMemory(k=3, return_messages=True)

# Simulate conversation turns
conversations = [
    ("Hi, my name is Sandeep", "Hello Sandeep! How can I help you today?"),
    ("I'm a Gen AI engineer", "Great! Gen AI is a fascinating field."),
    ("I use LangChain daily", "LangChain is excellent for building LLM apps."),
    ("What's my name?", "Based on our conversation, your name is Sandeep."),  # should remember
]

for human, ai in conversations:
    memory.save_context({"input": human}, {"output": ai})

print("Memory window (last 3 exchanges):")
history = memory.load_memory_variables({})["history"]
for msg in history:
    role = "Human" if msg.type == "human" else "AI"
    print(f"  [{role}]: {msg.content}")

---
## Q13. How do you evaluate a RAG pipeline? What metrics do you use?

**Answer:**
Use **RAGAS** framework — evaluates both retrieval and generation quality.

| Metric | Measures | Formula |
|---|---|---|
| **Faithfulness** | Answer grounded in context? | Claims in context / Total claims |
| **Answer Relevancy** | Answer relevant to question? | Semantic sim(generated_q, original_q) |
| **Context Precision** | Retrieved chunks useful? | Relevant chunks / Total chunks |
| **Context Recall** | All needed info retrieved? | Retrieved relevant / All relevant |

In [ ]:
# Manual RAGAS-style faithfulness check (simplified)
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

def check_faithfulness(answer: str, context_chunks: list) -> float:
    """Check if answer claims are supported by retrieved context."""
    answer_emb = model.encode(answer, convert_to_tensor=True)
    context_embs = model.encode(context_chunks, convert_to_tensor=True)
    scores = util.cos_sim(answer_emb, context_embs)[0]
    return float(scores.max())  # how well best context supports answer

def answer_relevancy(question: str, answer: str) -> float:
    """Check if answer is relevant to original question."""
    q_emb = model.encode(question, convert_to_tensor=True)
    a_emb = model.encode(answer, convert_to_tensor=True)
    return float(util.cos_sim(q_emb, a_emb)[0][0])

# Example
question = "What is the return policy?"
context = [
    "Customers can return products within 30 days for a full refund.",
    "All returns must include original packaging."
]
answer = "You can return your product within 30 days with original packaging for a full refund."

faith = check_faithfulness(answer, context)
relevancy = answer_relevancy(question, answer)

print(f"Faithfulness Score  : {faith:.4f}  (1.0 = fully grounded in context)")
print(f"Answer Relevancy    : {relevancy:.4f}  (1.0 = perfectly relevant to question)")
print(f"Overall Quality     : {(faith + relevancy) / 2:.4f}")

---
## Q14. What is the difference between `temperature`, `top_p`, and `top_k`?

**Answer:**

| Parameter | Controls | Effect |
|---|---|---|
| `temperature` | Randomness of token sampling | 0 = deterministic, 1 = creative, >1 = chaotic |
| `top_k` | Pool of K highest-prob tokens | Smaller K = more focused |
| `top_p` (nucleus) | Cumulative probability threshold | 0.9 = sample from top 90% prob mass |

**Production rule:** Use `temperature=0` for factual tasks, `0.7` for creative tasks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate effect of temperature on token probabilities
logits = np.array([3.0, 2.0, 1.5, 0.5, 0.2])  # raw logits for 5 tokens
tokens = ["good", "great", "fine", "bad", "terrible"]

def softmax_with_temp(logits, temperature):
    scaled = logits / temperature
    exp = np.exp(scaled - scaled.max())
    return exp / exp.sum()

temps = [0.1, 0.7, 1.0, 2.0]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle("Effect of Temperature on Token Probabilities", fontsize=13)

for ax, temp in zip(axes, temps):
    probs = softmax_with_temp(logits, temp)
    bars = ax.bar(tokens, probs, color=['#4A90D9', '#5BA85A', '#F5A623', '#E74C3C', '#9B59B6'])
    ax.set_title(f"Temperature = {temp}", fontsize=11)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=30)
    for bar, prob in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{prob:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('/home/claude/temperature_viz.png', dpi=100)
plt.show()
print("Low temp → peaked (deterministic) | High temp → flat (creative/random)")

---
## Q15. What is a Chain in LangChain? Explain LLMChain vs LCEL.

**Answer:**
A **Chain** connects components (prompt → LLM → parser) into a reusable pipeline.

- **LLMChain** (legacy): `chain = LLMChain(llm=..., prompt=...)`
- **LCEL** (modern): Uses `|` pipe operator, streaming-first, async-native

**LCEL is preferred** — composable, readable, supports batch/async natively.

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough

# LCEL Chain demonstration (without calling LLM — shows composition)
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant.\nUser question: {question}\nProvide a concise answer."
)

# Simulate the chain structure
def mock_llm(prompt_value):
    question = str(prompt_value).split("User question:")[1].split("\n")[0].strip()
    return f"[Mock LLM Response to: '{question}']"

# Build LCEL-style chain: prompt | llm | parser
mock_chain = prompt | RunnableLambda(mock_llm) | RunnableLambda(lambda x: x.strip())

result = mock_chain.invoke({"question": "What is RAG in AI?"})
print("LCEL Chain Result:")
print(result)

# Show chaining with context injection
print("\nLCEL Chain with context:")
context_chain = (
    RunnablePassthrough.assign(word_count=lambda x: len(x["question"].split()))
    | RunnableLambda(lambda x: f"Q({x['word_count']} words): {x['question']}")
)
print(context_chain.invoke({"question": "How does vector search work in RAG?"}))

---
## Q16. What is Token Limit / Context Window? How do you handle large documents?

**Answer:**
Context window = max tokens the model can process at once (input + output).

| Model | Context Window |
|---|---|
| GPT-3.5 | 16K tokens |
| GPT-4 | 128K tokens |
| Claude 3 | 200K tokens |
| Gemini 1.5 Pro | 1M tokens |

**Strategies for large docs:** Chunking + RAG, MapReduce chain, Refine chain, HyDE

In [ ]:
import tiktoken

def count_tokens(text: str, model: str = "gpt-3.5-turbo") -> int:
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

# MapReduce approach for long documents
def map_reduce_summarize(chunks: list) -> str:
    """
    MAP: Summarize each chunk individually
    REDUCE: Combine all summaries into final answer
    """
    # MAP phase
    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        tokens = count_tokens(chunk)
        summary = f"[Summary of chunk {i+1}: {chunk[:60]}...]"  # mock
        chunk_summaries.append((summary, tokens))
        print(f"  MAP - Chunk {i+1}: {tokens} tokens")
    
    # REDUCE phase
    combined = " ".join([s for s, _ in chunk_summaries])
    final_tokens = count_tokens(combined)
    print(f"\n  REDUCE - Combined summaries: {final_tokens} tokens")
    return f"FINAL SUMMARY: {combined[:150]}..."

# Test with sample chunks
sample_chunks = [
    "LangChain is a framework for building applications powered by language models. It provides standard interfaces for chains, agents, and memory.",
    "RAG combines retrieval systems with generative models to produce grounded, factual responses. It reduces hallucination significantly.",
    "LangGraph extends LangChain to build stateful multi-actor applications with cyclic computation graphs and built-in persistence.",
]

print("MapReduce Summarization:")
result = map_reduce_summarize(sample_chunks)
print(f"\n{result}")

---
## Q17. What is Hypothetical Document Embedding (HyDE)?

**Answer:**
HyDE improves retrieval by generating a **hypothetical answer** to the query, then using THAT as the search vector instead of the raw query.

**Why it works:** A hypothetical answer is semantically closer to real document chunks than the original short query.

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

# Corpus
documents = [
    "Neural networks learn through backpropagation and gradient descent.",
    "Transformers use self-attention to capture long-range dependencies in text.",
    "BERT is trained using masked language modeling on large text corpora.",
    "GPT models are autoregressive and trained to predict the next token.",
]

doc_embs = model.encode(documents, convert_to_tensor=True)

# Original query
query = "How are transformers trained?"

# HyDE: generate hypothetical answer (in production, LLM generates this)
hypothetical_answer = """Transformers are trained using large datasets with self-supervised 
objectives like masked language modeling or next token prediction using gradient descent."""

query_emb = model.encode(query, convert_to_tensor=True)
hyde_emb = model.encode(hypothetical_answer, convert_to_tensor=True)

print("Standard Query Retrieval:")
scores = util.cos_sim(query_emb, doc_embs)[0]
for doc, score in zip(documents, scores):
    print(f"  {score:.4f} | {doc[:60]}")

print("\nHyDE Retrieval (hypothetical answer as query):")
hyde_scores = util.cos_sim(hyde_emb, doc_embs)[0]
for doc, score in zip(documents, hyde_scores):
    print(f"  {score:.4f} | {doc[:60]}")

---
## Q18. What is the difference between Reranking and Retrieval?

**Answer:**
- **Retrieval** (Bi-encoder): Fast, approximate — embed query & docs separately, compare vectors
- **Reranking** (Cross-encoder): Slow, precise — process query + doc TOGETHER for exact relevance score

**Production pattern:** Retrieve top-50 with bi-encoder → Rerank to top-5 with cross-encoder

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder, util

bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
# cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')  # uncomment if installed

docs = [
    "Python is widely used in machine learning and data science",
    "LangChain is a framework for LLM application development",
    "Vector databases store high-dimensional embeddings for similarity search",
    "FastAPI is a modern Python web framework for building REST APIs",
    "RAG combines retrieval with generation to reduce LLM hallucinations",
]

query = "How do I reduce hallucinations in language models?"

# Stage 1: Bi-encoder retrieval (fast)
q_emb = bi_encoder.encode(query, convert_to_tensor=True)
d_embs = bi_encoder.encode(docs, convert_to_tensor=True)
bi_scores = util.cos_sim(q_emb, d_embs)[0]

ranked = sorted(zip(docs, bi_scores.tolist()), key=lambda x: x[1], reverse=True)

print("Stage 1 — Bi-encoder Retrieval (fast approximate):")
for doc, score in ranked:
    print(f"  {score:.4f} | {doc}")

print("\nStage 2 — Cross-encoder would then rerank top-3 with higher precision")
print("  (Cross-encoder processes query+doc jointly for precise scoring)")

---
## Q19. What is a System Prompt? How do you design one effectively?

**Answer:**
A system prompt sets the LLM's persona, constraints, tone, and output format — before any user interaction.

**Best practices:**
1. Define role clearly
2. Set explicit constraints (what NOT to do)
3. Specify output format
4. Add few-shot examples if needed
5. Keep concise — system prompt counts toward context window

In [ ]:
def build_system_prompt(role: str, domain: str, constraints: list, output_format: str) -> str:
    constraints_text = "\n".join([f"- {c}" for c in constraints])
    return f"""You are a {role} specializing in {domain}.

CONSTRAINTS:
{constraints_text}

OUTPUT FORMAT:
{output_format}

Always be concise, accurate, and professional."""

# Example: Customer support bot for a logistics company (NISSIN-inspired)
system_prompt = build_system_prompt(
    role="customer support specialist",
    domain="international logistics and shipment tracking",
    constraints=[
        "Only answer questions related to shipments, tracking, and delivery",
        "Never guess shipment status — only use provided tracking data",
        "If unsure, say 'I'll escalate this to our logistics team'",
        "Do not reveal internal system details or pricing formulas"
    ],
    output_format="""Always respond with:\n1. Direct answer\n2. Next steps (if any)\n3. ETA estimate (if applicable)"""
)

print("Engineered System Prompt:")
print("=" * 55)
print(system_prompt)
print(f"\nToken count: ~{len(system_prompt.split())} words")

---
## Q20. How would you design a Production RAG Pipeline end-to-end?

**Answer:**
This is the most common senior design question. Show you can think about the full system.

```
Data Pipeline: Documents → Clean → Chunk → Embed → Vector DB
Query Pipeline: Query → Embed → Retrieve → Rerank → Generate → Evaluate
```

**Production concerns:** caching, monitoring, async, fallback, guardrails

In [ ]:
from dataclasses import dataclass
from typing import List, Optional
import time

@dataclass
class RAGConfig:
    chunk_size: int = 512
    chunk_overlap: int = 64
    top_k_retrieve: int = 20     # broad retrieval
    top_k_rerank: int = 5        # precision after reranking
    min_relevance_score: float = 0.5
    cache_enabled: bool = True
    fallback_response: str = "I don't have enough information to answer this accurately."

class ProductionRAGPipeline:
    def __init__(self, config: RAGConfig):
        self.config = config
        self.cache = {}
        self.metrics = {"total_queries": 0, "cache_hits": 0, "fallbacks": 0}
    
    def retrieve(self, query: str) -> List[dict]:
        """Stage 1: Retrieve top-K chunks from vector DB"""
        # In prod: vectorstore.similarity_search_with_score(query, k=self.config.top_k_retrieve)
        return [{"content": f"Mock chunk {i}", "score": 0.9 - i*0.1} for i in range(3)]
    
    def rerank(self, query: str, chunks: List[dict]) -> List[dict]:
        """Stage 2: Rerank with cross-encoder for precision"""
        # In prod: cross_encoder.predict([(query, c['content']) for c in chunks])
        return sorted(chunks, key=lambda x: x["score"], reverse=True)[:self.config.top_k_rerank]
    
    def check_relevance(self, chunks: List[dict]) -> bool:
        """Stage 3: Filter out low-quality retrievals"""
        return any(c["score"] >= self.config.min_relevance_score for c in chunks)
    
    def generate(self, query: str, context: str) -> str:
        """Stage 4: Generate answer with LLM"""
        # In prod: llm.invoke(prompt.format(context=context, question=query))
        return f"[Generated answer for '{query}' using {len(context)} chars of context]"
    
    def query(self, user_query: str) -> dict:
        start = time.time()
        self.metrics["total_queries"] += 1
        
        # Cache check
        if self.config.cache_enabled and user_query in self.cache:
            self.metrics["cache_hits"] += 1
            return {**self.cache[user_query], "cached": True}
        
        # Pipeline execution
        chunks = self.retrieve(user_query)
        reranked = self.rerank(user_query, chunks)
        
        if not self.check_relevance(reranked):
            self.metrics["fallbacks"] += 1
            return {"answer": self.config.fallback_response, "confidence": 0.0}
        
        context = "\n".join([c["content"] for c in reranked])
        answer = self.generate(user_query, context)
        
        result = {
            "answer": answer,
            "sources": len(reranked),
            "top_score": reranked[0]["score"],
            "latency_ms": round((time.time() - start) * 1000, 2),
            "cached": False
        }
        self.cache[user_query] = result
        return result

# Test the pipeline
config = RAGConfig(top_k_retrieve=10, top_k_rerank=3)
pipeline = ProductionRAGPipeline(config)

queries = [
    "What is the return policy?",
    "How do I track my shipment?",
    "What is the return policy?",  # cache hit
]

for q in queries:
    result = pipeline.query(q)
    print(f"Q: {q}")
    print(f"   Answer   : {result['answer']}")
    print(f"   Cached   : {result.get('cached', False)}")
    print(f"   Latency  : {result.get('latency_ms', 'N/A')}ms\n")

print("Pipeline Metrics:", pipeline.metrics)

---
## 📊 Quick Reference Summary

| # | Topic | Key Concept |
|---|---|---|
| 1 | Semantic Search | Embeddings + cosine similarity |
| 2 | RAG | Retrieve → Inject → Generate |
| 3 | Embeddings | Dense vectors encoding meaning |
| 4 | Chunking | Split docs with overlap for context |
| 5 | Prompt Engineering | Zero-shot, Few-shot, CoT, ReAct |
| 6 | Agents | LLM + Tools + Reasoning loop |
| 7 | LangGraph | Stateful graph-based agent workflows |
| 8 | Fine-tuning vs RAG | Weights vs Knowledge update |
| 9 | Hallucination | Ground with RAG, temp=0, citation |
| 10 | Vector DBs | FAISS → local, Pinecone → prod |
| 11 | ReAct Pattern | Thought → Action → Observation loop |
| 12 | Memory | Buffer, Window, Summary, Vector |
| 13 | RAG Evaluation | Faithfulness, Relevancy, Precision |
| 14 | Temperature | 0 = deterministic, >0 = creative |
| 15 | LCEL | Pipe operator chains in LangChain |
| 16 | Context Window | Token limit → MapReduce for large docs |
| 17 | HyDE | Hypothetical answer improves retrieval |
| 18 | Reranking | Cross-encoder precision after bi-encoder |
| 19 | System Prompt | Role, constraints, format, examples |
| 20 | Production RAG | Retrieve → Rerank → Generate + cache/eval |

---
**Good luck with your Gen AI interviews! 🚀**